In [1]:
!pip install -U transformers peft bitsandbytes datasets trl accelerate

In [2]:
from huggingface_hub import login
login("")

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "google/gemma-3-1b-pt"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

print("Model loaded successfully")

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.00GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Model loaded successfully


In [ ]:
!pip install -q -U trl peft bitsandbytes datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.5/925.5 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 45.6 MB/s eta 0:00:00


In [4]:
from datasets import Dataset
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

dummy_rows = [
    {"instruction": "Vad är huvudstaden i Sverige?", "response": "Huvudstaden i Sverige är Stockholm."},
    {"instruction": "Skriv en kort hälsning på svenska.", "response": "Hej och välkommen! Så trevligt att ses."},
    {"instruction": "Vad heter den största sjön i Sverige?", "response": "Den största sjön i Sverige är Vänern."},
    {"instruction": "Ge ett svenskt ordspråk.", "response": "Övning ger färdighet."},
    {"instruction": "Vilken är den högsta bergstoppen i Sverige?", "response": "Den högsta bergstoppen i Sverige är Kebnekaise."},
    {"instruction": "Vad är valutan i Sverige?", "response": "Valutan i Sverige är svensk krona."},
    {"instruction": "Nämn tre stora städer i Sverige.", "response": "Tre stora städer i Sverige är Stockholm, Göteborg och Malmö."},
    {"instruction": "Vad heter Sveriges nationaldag?", "response": "Sveriges nationaldag firas den sjätte juni."},
    {"instruction": "Skriv en mening om hösten.", "response": "På hösten faller löven och vädret blir kallare."},
    {"instruction": "Vad är motsatsen till varm?", "response": "Motsatsen till varm är kall."},
    {"instruction": "Översätt ordet hund till engelska.", "response": "Ordet hund betyder dog på engelska."},
    {"instruction": "Vad heter havet öster om Sverige?", "response": "Havet öster om Sverige heter Östersjön."},
    {"instruction": "Ge ett tips för att lära sig svenska.", "response": "Ett bra tips är att läsa och lyssna på svenska varje dag."},
    {"instruction": "Vad är en älg?", "response": "En älg är ett stort däggdjur som lever i skogen."},
    {"instruction": "Skriv en kort mening om kaffe.", "response": "Många svenskar dricker gärna kaffe på fikapausen."},
    {"instruction": "Vad betyder ordet tack?", "response": "Ordet tack använder man för att visa uppskattning."},
]

def format_row(row):
    text = (
        "<start_of_turn>user\n"
        + row["instruction"]
        + "<end_of_turn>\n<start_of_turn>model\n"
        + row["response"]
        + "<end_of_turn>"
    )
    return {"text": text}

dataset = Dataset.from_list(dummy_rows).map(format_row)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

sft_config = SFTConfig(
    output_dir="gemma-lora-out",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=1,
    max_length=512,
    dataset_text_field="text",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=sft_config,
    peft_config=lora_config,
    processing_class=tokenizer,
)

trainer.train()
print("Training done")

Map:   0%|          | 0/16 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/16 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/16 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/16 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/16 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/16 [00:00<?, ? examples/s]

Step,Training Loss
1,8.331228
2,7.439558
3,7.441064
4,6.744339
5,6.706077
6,6.626276
7,6.332400
8,6.073606
9,5.733873
10,5.586827


Training done


In [5]:
prompt = "<start_of_turn>user\nVad är huvudstaden i Sverige?<end_of_turn>\n<start_of_turn>model\n"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
[transformers] Caching is incompatible with gradient checkpointing in Gemma3DecoderLayer. Setting `past_key_values=None`.


user
Vad är huvudstaden i Sverige?
model
VarVarVarVarVarVarVarVarVarVarVarVarVarVarVarVarVarVarVarVarVarVarVarVarVarVarVarVarVarVarVarVarVarVarVarVarVarVarVarVar
